In [2]:

!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 57.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

In [9]:
df = pd.read_excel("/content/ezh2_combined_dataset_nM_only.xlsx")

print(df.head())

       chembl_id                                   canonical_smiles  \
0  CHEMBL5206842  COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)...   
1  CHEMBL5188827  CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCOC...   
2  CHEMBL5172110  CSc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)c3c(c(C)c2C1=O...   
3  CHEMBL3264787  CC1(C)CC(NC(=O)c2ccc(Oc3cccc(-c4ccnnc4)c3C#N)c...   
4  CHEMBL5178022  CSc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(Cl)c2c(c1C)O[C...   

   original_IC50 original_IC50_unit  original_pIC50  IC50_nM IC50_unit  \
0         0.0097                 nM       11.013228   0.0097        nM   
1         0.0100                 nM       11.000000   0.0100        nM   
2         0.0280                 nM       10.552842   0.0280        nM   
3         0.0320                 nM       10.494850   0.0320        nM   
4         0.0380                 nM       10.420216   0.0380        nM   

   pIC50_normalized normalization_status       normalization_note  
0         11.013228            converted  Already reported i

In [10]:
df.shape

(3580, 10)

In [11]:
smiles = df.iloc[0]["canonical_smiles"]
print(smiles)

COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)N(Cc3c(C)cc(C)[nH]c3=O)CCC4)CC2)C1


In [12]:
mol = Chem.MolFromSmiles(smiles)

if mol is None:
    print("Invalid SMILES")
else:
    print("Valid SMILES")

Valid SMILES


In [13]:
molecular_weight = Descriptors.MolWt(mol)
logp = Crippen.MolLogP(mol)
hbd = Lipinski.NumHDonors(mol)
hba = Lipinski.NumHAcceptors(mol)

print("Molecular Weight:", molecular_weight)
print("LogP:", logp)
print("H-Bond Donors:", hbd)
print("H-Bond Acceptors:", hba)

Molecular Weight: 511.7320000000005
LogP: 4.943160000000005
H-Bond Donors: 1
H-Bond Acceptors: 5


In [14]:
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
    }

In [15]:
feature_list = []

for smiles in df["canonical_smiles"]:
    features = calculate_descriptors(smiles)
    feature_list.append(features)

In [16]:
feature_df = pd.DataFrame(feature_list)

print(feature_df.head())

     MolWt     LogP  HBD  HBA  RotatableBonds
0  511.732  4.94316    1    5               6
1  541.696  5.30844    2    5               8
2  546.133  5.17334    1    6               5
3  490.007  5.49998    2    6               5
4  520.095  4.90484    2    6               6


In [20]:
df = pd.concat([df, feature_df], axis=1)

print(df.head())

       chembl_id                                   canonical_smiles  \
0  CHEMBL5206842  COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)...   
1  CHEMBL5188827  CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCOC...   
2  CHEMBL5172110  CSc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)c3c(c(C)c2C1=O...   
3  CHEMBL3264787  CC1(C)CC(NC(=O)c2ccc(Oc3cccc(-c4ccnnc4)c3C#N)c...   
4  CHEMBL5178022  CSc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(Cl)c2c(c1C)O[C...   

   original_IC50 original_IC50_unit  original_pIC50  IC50_nM IC50_unit  \
0         0.0097                 nM       11.013228   0.0097        nM   
1         0.0100                 nM       11.000000   0.0100        nM   
2         0.0280                 nM       10.552842   0.0280        nM   
3         0.0320                 nM       10.494850   0.0320        nM   
4         0.0380                 nM       10.420216   0.0380        nM   

   pIC50_normalized normalization_status       normalization_note    MolWt  \
0         11.013228            converted  Already 

In [33]:
selected_columns = [
    "canonical_smiles",
    "IC50_nM",
    "pIC50_normalized",
    "MolWt",
    "LogP",
    "HBD",
    "HBA",
    "RotatableBonds"
]

df1 = df[selected_columns] # df should now be clean from cell 5c474be4
print(df1.head())

                                    canonical_smiles  IC50_nM  \
0  COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)...   0.0097   
1  CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCOC...   0.0100   
2  CSc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)c3c(c(C)c2C1=O...   0.0280   
3  CC1(C)CC(NC(=O)c2ccc(Oc3cccc(-c4ccnnc4)c3C#N)c...   0.0320   
4  CSc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(Cl)c2c(c1C)O[C...   0.0380   

   pIC50_normalized    MolWt     LogP  HBD  HBA  RotatableBonds  
0         11.013228  511.732  4.94316    1    5               6  
1         11.000000  541.696  5.30844    2    5               8  
2         10.552842  546.133  5.17334    1    6               5  
3         10.494850  490.007  5.49998    2    6               5  
4         10.420216  520.095  4.90484    2    6               6  


In [38]:
df1['Lipinski_Rule_of_5'] = (
    (df1['MolWt'] <= 500) &
    (df1['LogP'] <= 5) &
    (df1['HBD'] <= 5) &
    (df1['HBA'] <= 10)
)

display(df1.head())

/tmp/ipykernel_1356/2079718127.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['Lipinski_Rule_of_5'] = (


,canonical_smiles,IC50_nM,pIC50_normalized,MolWt,LogP,HBD,HBA,RotatableBonds,Lipinski_Rule_of_5
0,COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)...,0.0097,11.013228,511.732,4.94316,1,5,6,False
1,CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCOC...,0.0100,11.000000,541.696,5.30844,2,5,8,False
2,CSc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)c3c(c(C)c2C1=O...,0.0280,10.552842,546.133,5.17334,1,6,5,False
3,CC1(C)CC(NC(=O)c2ccc(Oc3cccc(-c4ccnnc4)c3C#N)c...,0.0320,10.494850,490.007,5.49998,2,6,5,False
4,CSc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(Cl)c2c(c1C)O[C...,0.0380,10.420216,520.095,4.90484,2,6,6,False


In [35]:
df.shape


(3580, 15)

In [39]:
lipinski_counts = df1['Lipinski_Rule_of_5'].value_counts()

lipinski_passed_smiles = df1[df1['Lipinski_Rule_of_5'] == True]['canonical_smiles']

display(lipinski_counts)
display(lipinski_passed_smiles.head())

,count
Lipinski_Rule_of_5,
False,2179
True,1401


,canonical_smiles
14,COC(c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[nH]c1...
16,COc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)cc([C@H](OC)C3...
17,COc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)cc(C(OC)C3CCOC...
18,COC(c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[nH]c1...
19,CO[C@@H](c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[...


In [48]:
final_df.to_csv('/content/final_df_with_lipinski_rules.csv', index=False)
print("DataFrame saved to '/content/final_df_with_lipinski_rules.csv'")

DataFrame saved to '/content/final_df_with_lipinski_rules.csv'


In [31]:
# Remove duplicate columns, keeping only the first occurrence
df = df.loc[:, ~df.columns.duplicated()]
print("DataFrame columns after removing duplicates:")
print(df.columns.tolist())

display(df.head())

DataFrame columns after removing duplicates:
['chembl_id', 'canonical_smiles', 'original_IC50', 'original_IC50_unit', 'original_pIC50', 'IC50_nM', 'IC50_unit', 'pIC50_normalized', 'normalization_status', 'normalization_note', 'MolWt', 'LogP', 'HBD', 'HBA', 'RotatableBonds']


,chembl_id,canonical_smiles,original_IC50,original_IC50_unit,original_pIC50,IC50_nM,IC50_unit,pIC50_normalized,normalization_status,normalization_note,MolWt,LogP,HBD,HBA,RotatableBonds
0,CHEMBL5206842,COC1CN([C@H]2CC[C@H]([C@@H](C)c3sc4c(c3C)C(=O)...,0.0097,nM,11.013228,0.0097,nM,11.013228,converted,Already reported in nM.,511.732,4.94316,1,5,6
1,CHEMBL5188827,CCCc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(-c2ccc(N3CCOC...,0.0100,nM,11.000000,0.0100,nM,11.000000,converted,Already reported in nM.,541.696,5.30844,2,5,8
2,CHEMBL5172110,CSc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)c3c(c(C)c2C1=O...,0.0280,nM,10.552842,0.0280,nM,10.552842,converted,Already reported in nM.,546.133,5.17334,1,6,5
3,CHEMBL3264787,CC1(C)CC(NC(=O)c2ccc(Oc3cccc(-c4ccnnc4)c3C#N)c...,0.0320,nM,10.494850,0.0320,nM,10.494850,converted,Already reported in nM.,490.007,5.49998,2,6,5
4,CHEMBL5178022,CSc1cc(C)[nH]c(=O)c1CNC(=O)c1cc(Cl)c2c(c1C)O[C...,0.0380,nM,10.420216,0.0380,nM,10.420216,converted,Already reported in nM.,520.095,4.90484,2,6,6


In [40]:
print(f"Number of compounds satisfying Lipinski's Rule of Five: {len(lipinski_passed_smiles)}")

df1 = df1[df1['Lipinski_Rule_of_5'] == True]
display(df1.head())

Number of compounds satisfying Lipinski's Rule of Five: 1401


,canonical_smiles,IC50_nM,pIC50_normalized,MolWt,LogP,HBD,HBA,RotatableBonds,Lipinski_Rule_of_5
14,COC(c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[nH]c1...,0.1,10.0,492.447,4.52634,1,4,5,True
16,COc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)cc([C@H](OC)C3...,0.1,10.0,467.349,3.53112,1,5,6,True
17,COc1cc(C)[nH]c(=O)c1CN1CCc2c(Cl)cc(C(OC)C3CCOC...,0.1,10.0,481.376,3.92122,1,5,6,True
18,COC(c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[nH]c1...,0.1,10.0,479.404,4.61114,1,4,5,True
19,CO[C@@H](c1cc(Cl)c2c(c1Cl)C(=O)N(Cc1c(C)cc(C)[...,0.1,10.0,451.350,3.83094,1,4,5,True


In [42]:
df1.shape

(1401, 9)

In [44]:
df1.to_csv('/content/df1_lipinski_filtered.csv', index=False)
print("Filtered DataFrame df1 saved to '/content/df1_lipinski_filtered.csv'")

Filtered DataFrame df1 saved to '/content/df1_lipinski_filtered.csv'
